# 🔬 Compare Dataset vs Live Observations

This notebook compares action chunks predicted from:
1. **Dataset observations** (recorded demonstrations)
2. **Live robot observations** (current robot state)

Use this to:
- Validate policy consistency between training data and deployment
- Debug observation preprocessing differences
- Understand how policy predictions differ based on input source

**Requirements:**
- Robot must be connected
- Dataset must be available locally

## 1. Configuration

In [ ]:
import pathlib

# ============================================================
# Policy checkpoint
# ============================================================
CHECKPOINT_DIR = pathlib.Path("/data/models/your_checkpoint_here")

# ============================================================
# Dataset settings
# ============================================================
DATASET_DIR = pathlib.Path("/data/lerobot/your_dataset_here")
EPISODE_INDEX = 0  # Which episode to compare against

# ============================================================
# Robot settings
# ============================================================
SERVER_ENDPOINT = "localhost:50051"

# ============================================================
# Common settings
# ============================================================
DEVICE = "cpu"  # or "cuda"
NUM_SAMPLES = 20  # Number of action chunk samples to generate per source

print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Dataset: {DATASET_DIR}")
print(f"Episode: {EPISODE_INDEX}")
print(f"Server: {SERVER_ENDPOINT}")
print(f"Device: {DEVICE}")
print(f"Samples per source: {NUM_SAMPLES}")

## 2. System Health Check

In [ ]:
import psutil
import shutil
import subprocess
import os

def check_system_health():
    """Comprehensive system health check for policy inference performance."""
    warnings = []
    
    print("=" * 70)
    print("🖥️  SYSTEM HEALTH CHECK FOR POLICY INFERENCE")
    print("=" * 70)
    
    # ================================================================
    # CPU Analysis
    # ================================================================
    print("\n" + "─" * 70)
    print("🔲 CPU STATUS")
    print("─" * 70)
    
    cpu_percent = psutil.cpu_percent(interval=0.5, percpu=True)
    cpu_count_logical = psutil.cpu_count(logical=True)
    cpu_count_physical = psutil.cpu_count(logical=False)
    cpu_freq = psutil.cpu_freq()
    load_avg = os.getloadavg()
    
    print(f"   Cores: {cpu_count_physical} physical, {cpu_count_logical} logical")
    print(f"   Usage: {sum(cpu_percent)/len(cpu_percent):.1f}% average")
    print(f"   Per-core: {[f'{p:.0f}%' for p in cpu_percent]}")
    
    if cpu_freq:
        freq_percent = (cpu_freq.current / cpu_freq.max * 100) if cpu_freq.max else 100
        print(f"   Frequency: {cpu_freq.current:.0f} MHz / {cpu_freq.max:.0f} MHz ({freq_percent:.0f}%)")
        if freq_percent < 70:
            warnings.append(f"⚠️ CPU frequency throttled to {freq_percent:.0f}%")
    
    print(f"   Load average (1/5/15 min): {load_avg[0]:.2f} / {load_avg[1]:.2f} / {load_avg[2]:.2f}")
    if load_avg[0] > cpu_count_logical:
        warnings.append(f"⚠️ System overloaded (load {load_avg[0]:.1f} > {cpu_count_logical} cores)")
    
    # Check CPU governor (Linux)
    try:
        with open('/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor', 'r') as f:
            governor = f.read().strip()
            print(f"   Governor: {governor}")
            if governor != 'performance':
                warnings.append(f"⚠️ CPU governor is '{governor}', not 'performance'")
    except:
        pass
    
    # ================================================================
    # Memory Analysis
    # ================================================================
    print("\n" + "─" * 70)
    print("💾 MEMORY STATUS")
    print("─" * 70)
    
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    print(f"   RAM: {mem.used / (1024**3):.1f} / {mem.total / (1024**3):.1f} GB ({mem.percent:.1f}% used)")
    print(f"   Available: {mem.available / (1024**3):.1f} GB")
    print(f"   Swap: {swap.used / (1024**3):.2f} / {swap.total / (1024**3):.1f} GB ({swap.percent:.1f}% used)")
    
    if mem.percent > 85:
        warnings.append(f"⚠️ High RAM usage: {mem.percent:.1f}%")
    if swap.percent > 10:
        warnings.append(f"⚠️ Swap in use: {swap.percent:.1f}% - may cause slowdowns")
    
    # ================================================================
    # Disk I/O Analysis
    # ================================================================
    print("\n" + "─" * 70)
    print("💿 DISK STATUS")
    print("─" * 70)
    
    disk = shutil.disk_usage("/")
    print(f"   Space: {disk.used / (1024**3):.1f} / {disk.total / (1024**3):.1f} GB ({100 * disk.used / disk.total:.1f}% used)")
    
    try:
        io_counters = psutil.disk_io_counters()
        print(f"   Read: {io_counters.read_bytes / (1024**3):.2f} GB total")
        print(f"   Write: {io_counters.write_bytes / (1024**3):.2f} GB total")
    except:
        pass
    
    # ================================================================
    # GPU Analysis (NVIDIA)
    # ================================================================
    print("\n" + "─" * 70)
    print("🎮 GPU STATUS")
    print("─" * 70)
    
    try:
        import torch
        print(f"   PyTorch version: {torch.__version__}")
        print(f"   CUDA available: {torch.cuda.is_available()}")
        
        if torch.cuda.is_available():
            print(f"   CUDA version: {torch.version.cuda}")
            print(f"   cuDNN version: {torch.backends.cudnn.version()}")
            print(f"   cuDNN enabled: {torch.backends.cudnn.enabled}")
            print(f"   cuDNN benchmark: {torch.backends.cudnn.benchmark}")
            
            if not torch.backends.cudnn.benchmark:
                warnings.append("⚠️ cuDNN benchmark mode disabled - enable for faster inference")
            
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                mem_allocated = torch.cuda.memory_allocated(i) / (1024**3)
                mem_reserved = torch.cuda.memory_reserved(i) / (1024**3)
                mem_total = props.total_memory / (1024**3)
                
                print(f"\n   GPU {i}: {props.name}")
                print(f"      Compute capability: {props.major}.{props.minor}")
                print(f"      Memory: {mem_allocated:.2f} GB allocated / {mem_reserved:.2f} GB reserved / {mem_total:.1f} GB total")
                
                # Detailed nvidia-smi query
                try:
                    result = subprocess.run(
                        ['nvidia-smi', '--query-gpu=temperature.gpu,power.draw,power.limit,clocks.current.graphics,clocks.max.graphics,clocks.current.memory,clocks.max.memory,pstate,utilization.gpu,utilization.memory,persistence_mode,performance_state',
                         '--format=csv,noheader,nounits', f'--id={i}'],
                        capture_output=True, text=True, timeout=5
                    )
                    if result.returncode == 0:
                        parts = [p.strip() for p in result.stdout.strip().split(',')]
                        if len(parts) >= 11:
                            temp, power_draw, power_limit, gpu_clock, gpu_clock_max, mem_clock, mem_clock_max, pstate, gpu_util, mem_util, persist_mode = parts[:11]
                            
                            print(f"      Temperature: {temp}°C")
                            print(f"      Power: {power_draw}W / {power_limit}W")
                            print(f"      GPU Clock: {gpu_clock} MHz / {gpu_clock_max} MHz")
                            print(f"      Memory Clock: {mem_clock} MHz / {mem_clock_max} MHz")
                            print(f"      Performance State: {pstate}")
                            print(f"      GPU Utilization: {gpu_util}%")
                            print(f"      Memory Utilization: {mem_util}%")
                            print(f"      Persistence Mode: {persist_mode}")
                            
                            # Check for issues
                            try:
                                if int(temp) > 80:
                                    warnings.append(f"⚠️ GPU {i} temperature high: {temp}°C (throttling may occur)")
                                if pstate not in ['P0', 'P1', 'P2']:
                                    warnings.append(f"⚠️ GPU {i} in low-power state: {pstate} (not P0/P1/P2)")
                                if float(gpu_clock) < float(gpu_clock_max) * 0.8:
                                    warnings.append(f"⚠️ GPU {i} clock throttled: {gpu_clock}/{gpu_clock_max} MHz")
                                if persist_mode == 'Disabled':
                                    warnings.append(f"⚠️ GPU {i} persistence mode disabled - first inference may be slow")
                            except:
                                pass
                except subprocess.TimeoutExpired:
                    print("      ⚠️ nvidia-smi timeout")
                except FileNotFoundError:
                    print("      ⚠️ nvidia-smi not found")
        else:
            print("   ⚠️ CUDA not available - running on CPU only")
            warnings.append("⚠️ No CUDA GPU - inference will be slower on CPU")
            
    except ImportError:
        print("   ⚠️ PyTorch not imported yet")
    
    # ================================================================
    # PyTorch Configuration
    # ================================================================
    print("\n" + "─" * 70)
    print("🔧 PYTORCH CONFIGURATION")
    print("─" * 70)
    
    try:
        import torch
        print(f"   Num threads: {torch.get_num_threads()}")
        print(f"   Num interop threads: {torch.get_num_interop_threads()}")
        
        # Check for common environment variables
        mkl_threads = os.environ.get('MKL_NUM_THREADS', 'not set')
        omp_threads = os.environ.get('OMP_NUM_THREADS', 'not set')
        print(f"   MKL_NUM_THREADS: {mkl_threads}")
        print(f"   OMP_NUM_THREADS: {omp_threads}")
        
        # Check inference mode hints
        print(f"   Default dtype: {torch.get_default_dtype()}")
        
        if torch.cuda.is_available():
            print(f"   TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
            print(f"   TF32 cuDNN: {torch.backends.cudnn.allow_tf32}")
    except:
        pass
    
    # ================================================================
    # Process Information
    # ================================================================
    print("\n" + "─" * 70)
    print("📊 CURRENT PROCESS")
    print("─" * 70)
    
    process = psutil.Process()
    print(f"   PID: {process.pid}")
    print(f"   Memory (RSS): {process.memory_info().rss / (1024**3):.2f} GB")
    print(f"   Memory (VMS): {process.memory_info().vms / (1024**3):.2f} GB")
    print(f"   CPU affinity: {process.cpu_affinity()}")
    print(f"   Threads: {process.num_threads()}")
    print(f"   Nice: {process.nice()}")
    
    if process.nice() > 0:
        warnings.append(f"⚠️ Process has low priority (nice={process.nice()})")
    
    # ================================================================
    # Network (for robot connection)
    # ================================================================
    print("\n" + "─" * 70)
    print("🌐 NETWORK STATUS")
    print("─" * 70)
    
    try:
        net_io = psutil.net_io_counters()
        print(f"   Bytes sent: {net_io.bytes_sent / (1024**2):.1f} MB")
        print(f"   Bytes recv: {net_io.bytes_recv / (1024**2):.1f} MB")
        print(f"   Packets dropped in: {net_io.dropin}")
        print(f"   Packets dropped out: {net_io.dropout}")
        
        if net_io.dropin > 0 or net_io.dropout > 0:
            warnings.append(f"⚠️ Network packet drops detected: {net_io.dropin} in, {net_io.dropout} out")
    except:
        pass
    
    # ================================================================
    # Power Profile (Linux)
    # ================================================================
    print("\n" + "─" * 70)
    print("⚡ POWER PROFILE")
    print("─" * 70)
    
    try:
        # Check power profile via powerprofilesctl
        result = subprocess.run(['powerprofilesctl', 'get'], capture_output=True, text=True, timeout=2)
        if result.returncode == 0:
            profile = result.stdout.strip()
            print(f"   Power profile: {profile}")
            if profile != 'performance':
                warnings.append(f"⚠️ Power profile is '{profile}', not 'performance'")
    except:
        try:
            # Fallback: check TLP or other
            with open('/sys/class/power_supply/AC/online', 'r') as f:
                on_ac = f.read().strip() == '1'
                print(f"   AC power: {'connected' if on_ac else 'disconnected (battery)'}")
                if not on_ac:
                    warnings.append("⚠️ Running on battery - performance may be limited")
        except:
            print("   Power profile: unknown")
    
    # ================================================================
    # Summary
    # ================================================================
    print("\n" + "=" * 70)
    if warnings:
        print("⚠️  WARNINGS DETECTED:")
        for w in warnings:
            print(f"   {w}")
    else:
        print("✅ ALL SYSTEMS OPTIMAL FOR INFERENCE")
    print("=" * 70)
    
    return warnings

# Run the check
health_warnings = check_system_health()

## 3. Load Policy

In [ ]:
from example_policies.robot_deploy.deploy_core.policy_manager import PolicyManager

# Load policy bundle
policy_bundle = PolicyManager.load_single(CHECKPOINT_DIR, DEVICE)

policy = policy_bundle.policy
cfg = policy_bundle.config
translator = policy_bundle.translator

n_obs_steps = getattr(policy.config, 'n_obs_steps', 1)
n_action_steps = getattr(policy.config, 'n_action_steps', 16)
action_mode = translator.action_mode

print(f"\n✅ Policy loaded!")
print(f"   n_obs_steps: {n_obs_steps}")
print(f"   n_action_steps: {n_action_steps}")
print(f"   action_mode: {action_mode}")

## 4. Load Dataset

In [ ]:
import os
import json
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset

os.environ["HF_HUB_OFFLINE"] = "1"

print(f"Loading dataset from {DATASET_DIR}...")
dataset = LeRobotDataset(
    repo_id=str(DATASET_DIR),
    root=DATASET_DIR,
    video_backend="pyav",
)
print(f"✅ Dataset loaded: {len(dataset)} frames")

# Get episode info
episodes_json = DATASET_DIR / "meta" / "episodes.jsonl"
episodes = []
with open(episodes_json, "r") as f:
    for line in f:
        episodes.append(json.loads(line))

episode_info = episodes[EPISODE_INDEX]
episode_length = episode_info["length"]
episode_start_idx = sum(episodes[i]["length"] for i in range(EPISODE_INDEX))

print(f"\n✅ Episode {EPISODE_INDEX}: {episode_length} frames")
print(f"   Dataset indices: {episode_start_idx} to {episode_start_idx + episode_length - 1}")

# Current frame counter (shared between dataset and comparison)
current_frame = 0

def get_observation_from_dataset(frame_idx):
    """Get observation dict from dataset at given frame index."""
    dataset_idx = episode_start_idx + frame_idx
    if dataset_idx >= episode_start_idx + episode_length:
        print(f"⚠️ Frame {frame_idx} exceeds episode length {episode_length}")
        return None
    
    sample = dataset[dataset_idx]
    observation = {}
    for key, value in sample.items():
        if key.startswith('observation.'):
            if torch.is_tensor(value):
                observation[key] = value.unsqueeze(0).to(DEVICE)
            else:
                observation[key] = value
    return observation

print(f"\n✅ Dataset ready!")

## 5. Connect to Robot

In [ ]:
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotInterface
from example_policies.robot_deploy.robot_io.robot_client import RobotClient

# Create persistent connection
robot_connection = RobotConnection(SERVER_ENDPOINT)
stub = robot_connection.__enter__()
robot_interface = RobotInterface(stub, cfg)

print(f"✅ Connected to robot at {SERVER_ENDPOINT}")

def get_observation_from_robot():
    """Get observation dict from live robot."""
    return robot_interface.get_observation(DEVICE)

## 6. Setup Visualization & Comparison Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from example_policies.utils.action_order import (
    ActionMode,
    DUAL_ABS_LEFT_POS_IDXS,
    DUAL_ABS_RIGHT_POS_IDXS,
    DUAL_DELTA_LEFT_POS_IDXS,
    DUAL_DELTA_RIGHT_POS_IDXS,
)


def prepare_observation(observation, expected_n_obs_steps):
    """Prepare observation for policy input.
    
    If the policy expects multiple observation steps but we only have one,
    we replicate the current observation to fill the expected sequence length.
    """
    obs_normalized = policy.normalize_inputs(observation)
    obs_normalized = dict(obs_normalized)
    
    # Fix state shape
    state = obs_normalized["observation.state"]
    if state.dim() == 2:
        state = state.unsqueeze(1)
    
    # Replicate state if we have fewer obs steps than expected
    if state.shape[1] < expected_n_obs_steps:
        # Repeat the last observation to fill the expected sequence length
        state = state[:, -1:, :].expand(-1, expected_n_obs_steps, -1)
    elif state.shape[1] > expected_n_obs_steps:
        state = state[:, -expected_n_obs_steps:, ...]
    obs_normalized["observation.state"] = state

    # ============================================================
    # FIX: Clamp normalized state to [-1, 1] to prevent OOD values
    # This prevents features with tiny normalization ranges (e.g., unused
    # gripper) from producing extreme values like +97 that break the model
    # ============================================================
    obs_normalized["observation.state"] = torch.clamp(
        obs_normalized["observation.state"], -1.0, 1.0
    )
    
    # Fix image shapes
    if policy.config.image_features:
        for key in policy.config.image_features:
            img = obs_normalized[key]
            if img.dim() == 4:
                img = img.unsqueeze(1)  # [B, C, H, W] -> [B, 1, C, H, W]
            
            # Replicate images if we have fewer obs steps than expected
            if img.shape[1] < expected_n_obs_steps:
                # Repeat the last frame to fill the expected sequence length
                img = img[:, -1:, ...].expand(-1, expected_n_obs_steps, -1, -1, -1)
            elif img.shape[1] > expected_n_obs_steps:
                img = img[:, -expected_n_obs_steps:, ...]
            obs_normalized[key] = img
        
        # Stack images
        image_tensors = [obs_normalized[key] for key in policy.config.image_features]
        obs_normalized["observation.images"] = torch.stack(image_tensors, dim=2)
    
    return obs_normalized


def generate_action_samples(observation, num_samples):
    """Generate multiple action chunk samples from an observation."""
    expected_n_obs_steps = policy.config.n_obs_steps
    obs_normalized = prepare_observation(observation, expected_n_obs_steps)
    
    with torch.inference_mode():
        global_cond = policy.dit_flow._prepare_global_conditioning(obs_normalized)
        
        actions_batch = policy.dit_flow.conditional_sample(
            batch_size=num_samples,
            global_cond=global_cond,
        )
        
        start = expected_n_obs_steps - 1
        end = start + policy.config.n_action_steps
        actions_batch = actions_batch[:, start:end]
        
        from example_policies.policies.models.dit_flow.modeling_dit_flow import ACTION
        actions_batch = policy.unnormalize_outputs({ACTION: actions_batch})[ACTION]
    
    return actions_batch


def extract_trajectory(action_chunk, observation, action_mode):
    """Extract left/right arm trajectories from action chunk."""
    chunk = action_chunk.detach().cpu().numpy()
    is_delta = action_mode == ActionMode.DELTA_TCP
    
    if is_delta:
        if observation is not None and 'observation.state' in observation:
            state_np = observation['observation.state'].squeeze(0).cpu()
            if state_np.dim() > 1:
                state_np = state_np[-1]
            state_np = state_np.numpy()
            left_init = state_np[:3]
            right_init = state_np[7:10]
        else:
            left_init = np.zeros(3)
            right_init = np.zeros(3)
        
        left_traj = [left_init.copy()]
        right_traj = [right_init.copy()]
        for t in range(chunk.shape[0]):
            left_traj.append(left_traj[-1] + chunk[t, DUAL_DELTA_LEFT_POS_IDXS])
            right_traj.append(right_traj[-1] + chunk[t, DUAL_DELTA_RIGHT_POS_IDXS])
        left_traj = np.array(left_traj)
        right_traj = np.array(right_traj)
    else:
        left_traj = chunk[:, DUAL_ABS_LEFT_POS_IDXS]
        right_traj = chunk[:, DUAL_ABS_RIGHT_POS_IDXS]
    
    return left_traj, right_traj


def show_images_comparison(obs_dataset, obs_robot):
    """Show side-by-side comparison of images from dataset and robot."""
    image_keys = sorted([k for k in obs_dataset.keys() if k.startswith('observation.image')])
    n = len(image_keys)
    
    fig, axes = plt.subplots(2, n, figsize=(4*n, 6))
    if n == 1:
        axes = axes.reshape(2, 1)
    
    for col, key in enumerate(image_keys):
        for row, (obs, label) in enumerate([(obs_dataset, 'Dataset'), (obs_robot, 'Robot')]):
            img = obs[key].squeeze(0).cpu()
            while img.dim() > 3:
                img = img[-1]
            if img.shape[0] in [1, 3]:
                img = img.permute(1, 2, 0)
            img = img.numpy()
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{label}: {key.split('.')[-1]}")
            axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()


def compare_states(obs_dataset, obs_robot):
    """Compare state vectors from dataset and robot."""
    state_d = obs_dataset['observation.state'].squeeze(0).cpu()
    state_r = obs_robot['observation.state'].squeeze(0).cpu()
    
    if state_d.dim() > 1:
        state_d = state_d[-1]
    if state_r.dim() > 1:
        state_r = state_r[-1]
    
    state_d = state_d.numpy()
    state_r = state_r.numpy()
    
    diff = np.abs(state_d - state_r)
    
    print(f"\n📊 State Comparison (dim={len(state_d)}):")
    print(f"   Dataset: {state_d[:8]}...")
    print(f"   Robot:   {state_r[:8]}...")
    print(f"   Diff:    {diff[:8]}...")
    print(f"\n   Mean abs diff: {diff.mean():.6f}")
    print(f"   Max abs diff:  {diff.max():.6f}")
    print(f"   Norm diff:     {np.linalg.norm(state_d - state_r):.6f}")
    
    return state_d, state_r, diff


print("✅ Visualization functions ready!")

## 7. Compare Observations

Get observations from both sources and compare them visually.

In [ ]:
# Set the frame to compare
current_frame = 0  # Change this to compare different timesteps

print(f"📍 Comparing at frame {current_frame}")
print("=" * 60)

# Get observations from both sources
obs_dataset = get_observation_from_dataset(current_frame)
obs_robot = get_observation_from_robot()

if obs_dataset is None or obs_robot is None:
    raise RuntimeError("Failed to get observations")

print(f"✅ Dataset observation: {list(obs_dataset.keys())}")
print(f"✅ Robot observation: {list(obs_robot.keys())}")

# Output directory for plots
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

PLOT_OUTPUT_DIR = pathlib.Path("outputs/comparison_plots")
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create PDF for observation comparison
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_path = PLOT_OUTPUT_DIR / f"observations_frame_{current_frame:04d}_{timestamp}.pdf"

with PdfPages(pdf_path) as pdf:
    # ============================================================
    # Page 1: Side-by-side image comparison
    # ============================================================
    image_keys = sorted([k for k in obs_dataset.keys() if k.startswith('observation.image')])
    n = len(image_keys)
    
    fig, axes = plt.subplots(2, n, figsize=(4*n, 6))
    if n == 1:
        axes = axes.reshape(2, 1)
    
    for col, key in enumerate(image_keys):
        for row, (obs, label) in enumerate([(obs_dataset, 'Dataset'), (obs_robot, 'Robot')]):
            img = obs[key].squeeze(0).cpu()
            while img.dim() > 3:
                img = img[-1]
            if img.shape[0] in [1, 3]:
                img = img.permute(1, 2, 0)
            img = img.numpy()
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{label}: {key.split('.')[-1]}")
            axes[row, col].axis('off')
    
    plt.suptitle(f'Image Comparison @ Frame {current_frame}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 2: State comparison
    # ============================================================
    state_d = obs_dataset['observation.state'].squeeze(0).cpu()
    state_r = obs_robot['observation.state'].squeeze(0).cpu()
    
    if state_d.dim() > 1:
        state_d = state_d[-1]
    if state_r.dim() > 1:
        state_r = state_r[-1]
    
    state_d = state_d.numpy()
    state_r = state_r.numpy()
    state_diff = np.abs(state_d - state_r)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot states
    x = np.arange(len(state_d))
    
    axes[0].bar(x, state_d, color='blue', alpha=0.7)
    axes[0].set_title('Dataset State', fontsize=12, fontweight='bold', color='blue')
    axes[0].set_xlabel('State Dimension')
    axes[0].set_ylabel('Value')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(x, state_r, color='red', alpha=0.7)
    axes[1].set_title('Robot State', fontsize=12, fontweight='bold', color='red')
    axes[1].set_xlabel('State Dimension')
    axes[1].set_ylabel('Value')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].bar(x, state_diff, color='purple', alpha=0.7)
    axes[2].set_title('Absolute Difference', fontsize=12, fontweight='bold', color='purple')
    axes[2].set_xlabel('State Dimension')
    axes[2].set_ylabel('|Dataset - Robot|')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'State Comparison @ Frame {current_frame}\nMean diff: {state_diff.mean():.6f}, Max diff: {state_diff.max():.6f}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 3: State values as text
    # ============================================================
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111)
    ax.axis('off')
    
    lines = []
    lines.append(f"STATE VALUES @ Frame {current_frame}")
    lines.append("=" * 80)
    lines.append(f"State dimension: {len(state_d)}")
    lines.append(f"Mean abs diff: {state_diff.mean():.6f}")
    lines.append(f"Max abs diff: {state_diff.max():.6f}")
    lines.append(f"Norm diff: {np.linalg.norm(state_d - state_r):.6f}")
    lines.append("")
    lines.append("─" * 80)
    lines.append("[BLUE] DATASET State:")
    lines.append("─" * 80)
    lines.append(np.array2string(state_d, precision=6, suppress_small=True, max_line_width=120))
    lines.append("")
    lines.append("─" * 80)
    lines.append("[RED] ROBOT State:")
    lines.append("─" * 80)
    lines.append(np.array2string(state_r, precision=6, suppress_small=True, max_line_width=120))
    lines.append("")
    lines.append("─" * 80)
    lines.append("[PURPLE] DIFFERENCE (Dataset - Robot):")
    lines.append("─" * 80)
    lines.append(np.array2string(state_d - state_r, precision=6, suppress_small=True, max_line_width=120))
    
    text = "\n".join(lines)
    ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=9, fontfamily='monospace',
            verticalalignment='top', horizontalalignment='left')
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    
    # ============================================================
    # Page 4: Image Embedding Comparison
    # ============================================================
    print("\n🖼️  Extracting image embeddings...")
    
    # Extract embeddings using the policy's RGB encoder
    dit_flow = policy.dit_flow
    use_separate = dit_flow.config.use_separate_rgb_encoder_per_camera
    
    embeddings_dataset = {}
    embeddings_robot = {}
    
    with torch.inference_mode():
        for idx, key in enumerate(image_keys):
            # Process dataset image
            img_d = obs_dataset[key]
            if img_d.dim() == 5:
                img_d = img_d[:, -1]
            elif img_d.dim() == 3:
                img_d = img_d.unsqueeze(0)
            
            # Process robot image
            img_r = obs_robot[key]
            if img_r.dim() == 5:
                img_r = img_r[:, -1]
            elif img_r.dim() == 3:
                img_r = img_r.unsqueeze(0)
            
            # Get encoder
            encoder = dit_flow.rgb_encoder[idx] if use_separate else dit_flow.rgb_encoder
            
            embeddings_dataset[key] = encoder(img_d).squeeze(0).cpu().numpy()
            embeddings_robot[key] = encoder(img_r).squeeze(0).cpu().numpy()
    
    # Combine embeddings
    combined_emb_d = np.concatenate([embeddings_dataset[k] for k in image_keys])
    combined_emb_r = np.concatenate([embeddings_robot[k] for k in image_keys])
    combined_emb_diff = np.abs(combined_emb_d - combined_emb_r)
    combined_cosine = np.dot(combined_emb_d, combined_emb_r) / (np.linalg.norm(combined_emb_d) * np.linalg.norm(combined_emb_r))
    
    # Create embedding comparison figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Bar plot of embedding values
    ax = axes[0, 0]
    x_emb = np.arange(len(combined_emb_d))
    width = 0.35
    ax.bar(x_emb - width/2, combined_emb_d, width, label='Dataset', alpha=0.7, color='blue')
    ax.bar(x_emb + width/2, combined_emb_r, width, label='Robot', alpha=0.7, color='red')
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('Value')
    ax.set_title('Image Embedding Values')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Difference plot
    ax = axes[0, 1]
    ax.bar(x_emb, combined_emb_diff, color='purple', alpha=0.7)
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('|Dataset - Robot|')
    ax.set_title(f'Absolute Difference (mean={combined_emb_diff.mean():.4f})')
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Scatter plot
    ax = axes[1, 0]
    ax.scatter(combined_emb_d, combined_emb_r, alpha=0.6, s=20)
    lims = [min(combined_emb_d.min(), combined_emb_r.min()), 
            max(combined_emb_d.max(), combined_emb_r.max())]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('Dataset Embedding')
    ax.set_ylabel('Robot Embedding')
    ax.set_title(f'Embedding Correlation (cosine={combined_cosine:.4f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    
    # Plot 4: Per-camera comparison
    ax = axes[1, 1]
    camera_names = [k.split('.')[-1] for k in image_keys]
    l2_dists = [np.linalg.norm(embeddings_dataset[k] - embeddings_robot[k]) for k in image_keys]
    cosine_sims = [np.dot(embeddings_dataset[k], embeddings_robot[k]) / 
                   (np.linalg.norm(embeddings_dataset[k]) * np.linalg.norm(embeddings_robot[k])) 
                   for k in image_keys]
    
    x_cam = np.arange(len(camera_names))
    ax2 = ax.twinx()
    bars1 = ax.bar(x_cam - 0.2, l2_dists, 0.4, label='L2 Distance', color='purple', alpha=0.7)
    bars2 = ax2.bar(x_cam + 0.2, cosine_sims, 0.4, label='Cosine Sim', color='green', alpha=0.7)
    ax.set_xlabel('Camera')
    ax.set_ylabel('L2 Distance', color='purple')
    ax2.set_ylabel('Cosine Similarity', color='green')
    ax.set_xticks(x_cam)
    ax.set_xticklabels(camera_names, rotation=45, ha='right')
    ax.set_title('Per-Camera Embedding Similarity')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Image Embedding Comparison @ Frame {current_frame}\nCombined: L2={np.linalg.norm(combined_emb_d - combined_emb_r):.4f}, Cosine={combined_cosine:.4f}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 5: Embedding values as text
    # ============================================================
    fig = plt.figure(figsize=(14, 12))
    ax = fig.add_subplot(111)
    ax.axis('off')
    
    emb_lines = []
    emb_lines.append(f"IMAGE EMBEDDING VALUES @ Frame {current_frame}")
    emb_lines.append("=" * 80)
    emb_lines.append(f"Combined embedding dim: {len(combined_emb_d)}")
    emb_lines.append(f"Number of cameras: {len(image_keys)}")
    emb_lines.append(f"Combined L2 distance: {np.linalg.norm(combined_emb_d - combined_emb_r):.6f}")
    emb_lines.append(f"Combined cosine similarity: {combined_cosine:.6f}")
    emb_lines.append("")
    
    for key in image_keys:
        camera_name = key.split('.')[-1]
        emb_d = embeddings_dataset[key]
        emb_r = embeddings_robot[key]
        diff = np.abs(emb_d - emb_r)
        cos_sim = np.dot(emb_d, emb_r) / (np.linalg.norm(emb_d) * np.linalg.norm(emb_r))
        
        emb_lines.append("─" * 80)
        emb_lines.append(f"[{camera_name.upper()}] Dim={len(emb_d)}, L2={np.linalg.norm(emb_d - emb_r):.4f}, Cosine={cos_sim:.4f}")
        emb_lines.append("─" * 80)
        emb_lines.append(f"  Dataset: {np.array2string(emb_d[:16], precision=4, suppress_small=True)}...")
        emb_lines.append(f"  Robot:   {np.array2string(emb_r[:16], precision=4, suppress_small=True)}...")
        emb_lines.append("")
    
    emb_text = "\n".join(emb_lines)
    ax.text(0.02, 0.98, emb_text, transform=ax.transAxes, fontsize=8, fontfamily='monospace',
            verticalalignment='top', horizontalalignment='left')
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

print(f"\n💾 Saved observation report: {pdf_path}")

# Also print state comparison to console
print(f"\n📊 State Comparison (dim={len(state_d)}):")
print(f"   Dataset: {state_d[:8]}...")
print(f"   Robot:   {state_r[:8]}...")
print(f"   Diff:    {state_diff[:8]}...")
print(f"\n   Mean abs diff: {state_diff.mean():.6f}")
print(f"   Max abs diff:  {state_diff.max():.6f}")
print(f"   Norm diff:     {np.linalg.norm(state_d - state_r):.6f}")

## 7b. Save Observations for Transfer

Save the dataset and robot observations to a file for reproducing on another machine.

In [ ]:
# Save observations to a single file for transfer to another machine
from datetime import datetime

# Create filename with metadata (keep it simple for JupyterHub compatibility)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_filename = f"obs_f{current_frame:04d}_e{EPISODE_INDEX}_{timestamp}.pt"

# Save in the same directory as the notebook (no absolute path needed)
save_path = pathlib.Path(save_filename)

# Convert observations to CPU tensors for portability
def obs_to_cpu(obs):
    """Convert all tensors in observation dict to CPU."""
    return {k: v.cpu() if torch.is_tensor(v) else v for k, v in obs.items()}

# Build the data bundle
data_bundle = {
    # Metadata
    "metadata": {
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "dataset_dir": str(DATASET_DIR),
        "episode_index": EPISODE_INDEX,
        "frame_index": current_frame,
        "episode_length": episode_length,
        "timestamp": timestamp,
        "action_mode": str(action_mode),
        "n_obs_steps": n_obs_steps,
        "n_action_steps": n_action_steps,
    },
    # Observations
    "obs_dataset": obs_to_cpu(obs_dataset),
    "obs_robot": obs_to_cpu(obs_robot),
}

# Save
torch.save(data_bundle, save_path)

# Verify the file was saved correctly
import os
actual_size = os.path.getsize(save_filename)

print(f"✅ Saved observations to: {save_filename}")
print(f"   File size: {actual_size / (1024*1024):.2f} MB")
print(f"\n📋 Contents:")
print(f"   - Metadata: checkpoint, dataset, episode {EPISODE_INDEX}, frame {current_frame}")
print(f"   - obs_dataset keys: {list(obs_dataset.keys())}")
print(f"   - obs_robot keys: {list(obs_robot.keys())}")
print(f"\n💡 Download this .pt file and use cell 7c to load it on another machine.")

## 7c. Load Observations from File

Load previously saved observations to reproduce inference on another machine (or skip live robot connection).

In [ ]:
# ============================================================
# LOAD OBSERVATIONS FROM FILE
# ============================================================
import torch
import pathlib
import numpy as np
import matplotlib.pyplot as plt

# Set the filename of your saved observations (same folder as notebook)
LOAD_FILENAME = "obs_f0000_e0_20260123_092421.pt"
# ^^^ Update this to your actual filename ^^^

DEVICE = "cpu"  # or "cuda"

LOAD_PATH = pathlib.Path(LOAD_FILENAME)

# Load the data bundle
data_bundle = torch.load(LOAD_PATH, map_location=DEVICE, weights_only=False)

# Extract metadata
metadata = data_bundle["metadata"]
print("=" * 60)
print("📂 LOADED OBSERVATIONS FROM FILE")
print("=" * 60)
print(f"   Source checkpoint: {metadata['checkpoint_dir']}")
print(f"   Source dataset: {metadata['dataset_dir']}")
print(f"   Episode: {metadata['episode_index']}, Frame: {metadata['frame_index']}")
print(f"   Action mode: {metadata['action_mode']}")
print(f"   Saved at: {metadata['timestamp']}")
print("=" * 60)

# Convert to device
def obs_to_device(obs, device):
    """Move all tensors to specified device."""
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in obs.items()}

# Load observations (these will override the live ones)
obs_dataset = obs_to_device(data_bundle["obs_dataset"], DEVICE)
obs_robot = obs_to_device(data_bundle["obs_robot"], DEVICE)

# Update frame info from loaded data
current_frame = metadata["frame_index"]

print(f"\n✅ Loaded observations to '{DEVICE}'")
print(f"   obs_dataset keys: {list(obs_dataset.keys())}")
print(f"   obs_robot keys: {list(obs_robot.keys())}")

# ============================================================
# Visualize loaded observations (same as cell 7)
# ============================================================
print("\n" + "=" * 60)
print(f"📍 Visualizing loaded observations @ frame {current_frame}")
print("=" * 60)

# Side-by-side image comparison
image_keys = sorted([k for k in obs_dataset.keys() if k.startswith('observation.image')])
n = len(image_keys)

if n > 0:
    fig, axes = plt.subplots(2, n, figsize=(4*n, 6))
    if n == 1:
        axes = axes.reshape(2, 1)
    
    for col, key in enumerate(image_keys):
        for row, (obs, label) in enumerate([(obs_dataset, 'Dataset'), (obs_robot, 'Robot')]):
            img = obs[key].squeeze(0).cpu()
            while img.dim() > 3:
                img = img[-1]
            if img.shape[0] in [1, 3]:
                img = img.permute(1, 2, 0)
            img = img.numpy()
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{label}: {key.split('.')[-1]}")
            axes[row, col].axis('off')
    
    plt.suptitle(f'Image Comparison @ Frame {current_frame} (from file)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# State comparison
state_d = obs_dataset['observation.state'].squeeze(0).cpu()
state_r = obs_robot['observation.state'].squeeze(0).cpu()

if state_d.dim() > 1:
    state_d = state_d[-1]
if state_r.dim() > 1:
    state_r = state_r[-1]

state_d = state_d.numpy()
state_r = state_r.numpy()
state_diff = np.abs(state_d - state_r)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

x = np.arange(len(state_d))

axes[0].bar(x, state_d, color='blue', alpha=0.7)
axes[0].set_title('Dataset State', fontsize=12, fontweight='bold', color='blue')
axes[0].set_xlabel('State Dimension')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

axes[1].bar(x, state_r, color='red', alpha=0.7)
axes[1].set_title('Robot State', fontsize=12, fontweight='bold', color='red')
axes[1].set_xlabel('State Dimension')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

axes[2].bar(x, state_diff, color='purple', alpha=0.7)
axes[2].set_title('Absolute Difference', fontsize=12, fontweight='bold', color='purple')
axes[2].set_xlabel('State Dimension')
axes[2].set_ylabel('|Dataset - Robot|')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'State Comparison @ Frame {current_frame} (from file)\nMean diff: {state_diff.mean():.6f}, Max diff: {state_diff.max():.6f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print state comparison to console
print(f"\n📊 State Comparison (dim={len(state_d)}):")
print(f"   Dataset: {state_d[:8]}...")
print(f"   Robot:   {state_r[:8]}...")
print(f"   Diff:    {state_diff[:8]}...")
print(f"\n   Mean abs diff: {state_diff.mean():.6f}")
print(f"   Max abs diff:  {state_diff.max():.6f}")
print(f"   Norm diff:     {np.linalg.norm(state_d - state_r):.6f}")

# ============================================================
# IMAGE EMBEDDING COMPARISON
# ============================================================
# This requires loading the policy to get the RGB encoder
# If policy is not available, we skip this section

try:
    # Check if policy exists in global scope
    if 'policy' not in dir():
        print("\n⚠️ Policy not loaded - skipping image embedding comparison.")
        print("   To compare embeddings, first run cells 1-4 to load the policy.")
        raise NameError("policy not defined")
    
    print("\n" + "=" * 60)
    print("🔍 IMAGE EMBEDDING COMPARISON")
    print("=" * 60)
    
    # Extract the RGB encoder from the policy
    rgb_encoder = policy.model.rgb_encoder
    rgb_encoder.eval()
    
    # Prepare images for encoding
    embedding_results = {}
    
    for key in image_keys:
        # Get images from both sources
        img_dataset = obs_dataset[key].clone()
        img_robot = obs_robot[key].clone()
        
        # Ensure proper shape: (batch, channels, height, width)
        if img_dataset.dim() == 5:  # (batch, n_obs, C, H, W)
            img_dataset = img_dataset[:, -1]  # Take last observation
        if img_robot.dim() == 5:
            img_robot = img_robot[:, -1]
        
        # Move to same device as encoder
        encoder_device = next(rgb_encoder.parameters()).device
        img_dataset = img_dataset.to(encoder_device)
        img_robot = img_robot.to(encoder_device)
        
        # Get embeddings
        with torch.no_grad():
            emb_dataset = rgb_encoder(img_dataset)
            emb_robot = rgb_encoder(img_robot)
        
        # Compute differences
        emb_dataset_np = emb_dataset.squeeze().cpu().numpy()
        emb_robot_np = emb_robot.squeeze().cpu().numpy()
        
        l2_dist = np.linalg.norm(emb_dataset_np - emb_robot_np)
        cosine_sim = np.dot(emb_dataset_np, emb_robot_np) / (
            np.linalg.norm(emb_dataset_np) * np.linalg.norm(emb_robot_np) + 1e-8
        )
        
        embedding_results[key] = {
            'dataset': emb_dataset_np,
            'robot': emb_robot_np,
            'l2_distance': l2_dist,
            'cosine_similarity': cosine_sim,
        }
        
        cam_name = key.split('.')[-1]
        print(f"\n   📷 {cam_name}:")
        print(f"      Embedding dim: {len(emb_dataset_np)}")
        print(f"      L2 distance:   {l2_dist:.6f}")
        print(f"      Cosine sim:    {cosine_sim:.6f}")
    
    # Plot embedding comparison
    n_cameras = len(embedding_results)
    fig, axes = plt.subplots(n_cameras, 3, figsize=(15, 4*n_cameras))
    if n_cameras == 1:
        axes = axes.reshape(1, -1)
    
    for row, (key, result) in enumerate(embedding_results.items()):
        cam_name = key.split('.')[-1]
        emb_d = result['dataset']
        emb_r = result['robot']
        emb_diff = np.abs(emb_d - emb_r)
        
        x = np.arange(len(emb_d))
        
        axes[row, 0].bar(x, emb_d, color='blue', alpha=0.7, width=0.8)
        axes[row, 0].set_title(f'{cam_name}: Dataset Embedding', fontweight='bold', color='blue')
        axes[row, 0].set_xlabel('Dimension')
        axes[row, 0].set_ylabel('Value')
        axes[row, 0].grid(True, alpha=0.3)
        
        axes[row, 1].bar(x, emb_r, color='red', alpha=0.7, width=0.8)
        axes[row, 1].set_title(f'{cam_name}: Robot Embedding', fontweight='bold', color='red')
        axes[row, 1].set_xlabel('Dimension')
        axes[row, 1].set_ylabel('Value')
        axes[row, 1].grid(True, alpha=0.3)
        
        axes[row, 2].bar(x, emb_diff, color='purple', alpha=0.7, width=0.8)
        axes[row, 2].set_title(f'{cam_name}: |Diff| (L2={result["l2_distance"]:.4f}, cos={result["cosine_similarity"]:.4f})', 
                               fontweight='bold', color='purple')
        axes[row, 2].set_xlabel('Dimension')
        axes[row, 2].set_ylabel('|Dataset - Robot|')
        axes[row, 2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Image Embedding Comparison @ Frame {current_frame} (from file)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Combined embedding analysis
    all_emb_dataset = np.concatenate([r['dataset'] for r in embedding_results.values()])
    all_emb_robot = np.concatenate([r['robot'] for r in embedding_results.values()])
    combined_l2 = np.linalg.norm(all_emb_dataset - all_emb_robot)
    combined_cos = np.dot(all_emb_dataset, all_emb_robot) / (
        np.linalg.norm(all_emb_dataset) * np.linalg.norm(all_emb_robot) + 1e-8
    )
    
    print(f"\n📊 Combined Embedding Analysis:")
    print(f"   Total embedding dim: {len(all_emb_dataset)}")
    print(f"   Combined L2 distance: {combined_l2:.6f}")
    print(f"   Combined cosine sim:  {combined_cos:.6f}")

except NameError:
    pass  # Policy not loaded, skip embedding comparison
except Exception as e:
    print(f"\n⚠️ Error computing image embeddings: {e}")
    import traceback
    traceback.print_exc()

print(f"\n💡 Now run the inference cells (8, 9, 10) to analyze these loaded observations.")

## 7d. Compare Normalized States

Visualize how the raw states look AFTER normalization (what the neural network actually sees).
This is critical because features with tiny normalization ranges can produce extreme values.

In [ ]:
# ============================================================
# COMPARE NORMALIZED STATES
# ============================================================
# This shows what the neural network actually "sees" after normalization
# Features with tiny normalization ranges can produce extreme values!

print("=" * 60)
print("📊 NORMALIZED STATE COMPARISON")
print("=" * 60)

# Get normalized observations using the policy's normalize_inputs
obs_dataset_norm = policy.normalize_inputs(obs_dataset)
obs_robot_norm = policy.normalize_inputs(obs_robot)

# Extract normalized states
state_d_norm = obs_dataset_norm['observation.state'].squeeze(0).cpu()
state_r_norm = obs_robot_norm['observation.state'].squeeze(0).cpu()

if state_d_norm.dim() > 1:
    state_d_norm = state_d_norm[-1]
if state_r_norm.dim() > 1:
    state_r_norm = state_r_norm[-1]

state_d_norm = state_d_norm.numpy()
state_r_norm = state_r_norm.numpy()
state_norm_diff = np.abs(state_d_norm - state_r_norm)

# Check for out-of-distribution values
ood_mask_d = np.abs(state_d_norm) > 1.0
ood_mask_r = np.abs(state_r_norm) > 1.0
n_ood_d = np.sum(ood_mask_d)
n_ood_r = np.sum(ood_mask_r)

print(f"\n⚠️  OUT-OF-DISTRIBUTION CHECK (normalized values should be in [-1, 1]):")
print(f"   Dataset: {n_ood_d} features out of range")
print(f"   Robot:   {n_ood_r} features out of range")

if n_ood_d > 0:
    print(f"   Dataset OOD indices: {np.where(ood_mask_d)[0].tolist()}")
    print(f"   Dataset OOD values:  {state_d_norm[ood_mask_d].tolist()}")
if n_ood_r > 0:
    print(f"   Robot OOD indices:   {np.where(ood_mask_r)[0].tolist()}")
    print(f"   Robot OOD values:    {state_r_norm[ood_mask_r].tolist()}")

# Plot normalized states
fig, axes = plt.subplots(4, 1, figsize=(16, 14))

x = np.arange(len(state_d_norm))

# Dataset normalized state
bars_d = axes[0].bar(x, state_d_norm, color='blue', alpha=0.7)
# Highlight OOD values in orange
for i in np.where(ood_mask_d)[0]:
    bars_d[i].set_color('orange')
axes[0].axhline(y=1.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Expected range [-1, 1]')
axes[0].axhline(y=-1.0, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0].set_title('Dataset NORMALIZED State (should be in [-1, 1])', fontsize=12, fontweight='bold', color='blue')
axes[0].set_xlabel('State Dimension')
axes[0].set_ylabel('Normalized Value')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Robot normalized state
bars_r = axes[1].bar(x, state_r_norm, color='red', alpha=0.7)
# Highlight OOD values in orange
for i in np.where(ood_mask_r)[0]:
    bars_r[i].set_color('orange')
axes[1].axhline(y=1.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Expected range [-1, 1]')
axes[1].axhline(y=-1.0, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[1].set_title('Robot NORMALIZED State (should be in [-1, 1])', fontsize=12, fontweight='bold', color='red')
axes[1].set_xlabel('State Dimension')
axes[1].set_ylabel('Normalized Value')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Difference in normalized states
axes[2].bar(x, state_norm_diff, color='purple', alpha=0.7)
axes[2].set_title('Absolute Difference in NORMALIZED States', fontsize=12, fontweight='bold', color='purple')
axes[2].set_xlabel('State Dimension')
axes[2].set_ylabel('|Dataset - Robot|')
axes[2].grid(True, alpha=0.3)

# Show the worst offenders
worst_indices = np.argsort(state_norm_diff)[-5:][::-1]
worst_text = "Top 5 largest normalized differences:\n"
for idx in worst_indices:
    worst_text += f"  [dim {idx}]: diff={state_norm_diff[idx]:.2f}, dataset={state_d_norm[idx]:.2f}, robot={state_r_norm[idx]:.2f}\n"

axes[3].axis('off')
axes[3].text(0.02, 0.95, worst_text, transform=axes[3].transAxes, fontsize=11, fontfamily='monospace',
             verticalalignment='top', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Add summary
summary_text = f"""
SUMMARY:
  Mean normalized diff:  {state_norm_diff.mean():.4f}
  Max normalized diff:   {state_norm_diff.max():.4f}
  Max |dataset|:         {np.abs(state_d_norm).max():.4f}
  Max |robot|:           {np.abs(state_r_norm).max():.4f}
  Dataset OOD features:  {n_ood_d}
  Robot OOD features:    {n_ood_r}
"""
axes[3].text(0.5, 0.95, summary_text, transform=axes[3].transAxes, fontsize=11, fontfamily='monospace',
             verticalalignment='top', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.suptitle(f'NORMALIZED State Comparison @ Frame {current_frame}\n(Orange bars = OUT OF DISTRIBUTION values)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n💡 If you see values far outside [-1, 1], those features are out-of-distribution!")
print(f"   This typically happens when features have tiny normalization ranges (e.g., unused gripper).")
print(f"   FIX: Clamp normalized states to [-1, 1] before feeding to the model.")

## 8. Compare Action Chunk Predictions

Generate action chunks from both sources and compare them.

In [ ]:
policy.reset()

print(f"🎲 Generating {NUM_SAMPLES} action samples from each source...")

# Generate samples from dataset observation
actions_dataset = generate_action_samples(obs_dataset, NUM_SAMPLES)
print(f"   Dataset samples: {actions_dataset.shape}")

# Generate samples from robot observation  
actions_robot = generate_action_samples(obs_robot, NUM_SAMPLES)
print(f"   Robot samples: {actions_robot.shape}")

# ============================================================
# Create HYBRID observations
# ============================================================
print("\n🔀 Creating hybrid observations...")

# Hybrid 1: Dataset images + Robot state
obs_hybrid_dataset_img_robot_state = {}
for key in obs_dataset.keys():
    if key.startswith('observation.image'):
        obs_hybrid_dataset_img_robot_state[key] = obs_dataset[key].clone()
    elif key == 'observation.state':
        obs_hybrid_dataset_img_robot_state[key] = obs_robot[key].clone()
    else:
        obs_hybrid_dataset_img_robot_state[key] = obs_dataset[key]

# Hybrid 2: Robot images + Dataset state
obs_hybrid_robot_img_dataset_state = {}
for key in obs_robot.keys():
    if key.startswith('observation.image'):
        obs_hybrid_robot_img_dataset_state[key] = obs_robot[key].clone()
    elif key == 'observation.state':
        obs_hybrid_robot_img_dataset_state[key] = obs_dataset[key].clone()
    else:
        obs_hybrid_robot_img_dataset_state[key] = obs_robot[key]

# Generate samples from hybrid observations
actions_hybrid1 = generate_action_samples(obs_hybrid_dataset_img_robot_state, NUM_SAMPLES)
print(f"   Hybrid1 (Dataset img + Robot state): {actions_hybrid1.shape}")

actions_hybrid2 = generate_action_samples(obs_hybrid_robot_img_dataset_state, NUM_SAMPLES)
print(f"   Hybrid2 (Robot img + Dataset state): {actions_hybrid2.shape}")

# Output directory for plots
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

PLOT_OUTPUT_DIR = pathlib.Path("outputs/comparison_plots")
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create PDF for this comparison
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_path = PLOT_OUTPUT_DIR / f"comparison_frame_{current_frame:04d}_{timestamp}.pdf"

with PdfPages(pdf_path) as pdf:
    # ============================================================
    # Page 1: 4-panel comparison plot (original sources)
    # ============================================================
    fig = plt.figure(figsize=(16, 12))
    
    ax1 = fig.add_subplot(221, projection='3d')
    ax2 = fig.add_subplot(222, projection='3d')
    ax3 = fig.add_subplot(223, projection='3d')
    ax4 = fig.add_subplot(224, projection='3d')
    
    # Dataset samples - Left arm
    for i in range(NUM_SAMPLES):
        left_traj, _ = extract_trajectory(actions_dataset[i], obs_dataset, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax1.plot(left_traj[:, 0], left_traj[:, 1], left_traj[:, 2], 
                 'b-o' if i == 0 else 'b-', linewidth=lw, markersize=2, alpha=alpha)
    ax1.set_title('Dataset - Left Arm', fontsize=12, fontweight='bold', color='blue')
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    
    # Dataset samples - Right arm
    for i in range(NUM_SAMPLES):
        _, right_traj = extract_trajectory(actions_dataset[i], obs_dataset, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax2.plot(right_traj[:, 0], right_traj[:, 1], right_traj[:, 2], 
                 'b-o' if i == 0 else 'b-', linewidth=lw, markersize=2, alpha=alpha)
    ax2.set_title('Dataset - Right Arm', fontsize=12, fontweight='bold', color='blue')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    
    # Robot samples - Left arm
    for i in range(NUM_SAMPLES):
        left_traj, _ = extract_trajectory(actions_robot[i], obs_robot, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax3.plot(left_traj[:, 0], left_traj[:, 1], left_traj[:, 2], 
                 'r-o' if i == 0 else 'r-', linewidth=lw, markersize=2, alpha=alpha)
    ax3.set_title('Robot - Left Arm', fontsize=12, fontweight='bold', color='red')
    ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')
    
    # Robot samples - Right arm
    for i in range(NUM_SAMPLES):
        _, right_traj = extract_trajectory(actions_robot[i], obs_robot, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax4.plot(right_traj[:, 0], right_traj[:, 1], right_traj[:, 2], 
                 'r-o' if i == 0 else 'r-', linewidth=lw, markersize=2, alpha=alpha)
    ax4.set_title('Robot - Right Arm', fontsize=12, fontweight='bold', color='red')
    ax4.set_xlabel('X'); ax4.set_ylabel('Y'); ax4.set_zlabel('Z')
    
    plt.suptitle(f'Action Chunk Comparison @ Frame {current_frame}\nBlue=Dataset, Red=Robot ({NUM_SAMPLES} samples each)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 2: 4-panel HYBRID comparison plot
    # ============================================================
    fig = plt.figure(figsize=(16, 12))
    
    ax1 = fig.add_subplot(221, projection='3d')
    ax2 = fig.add_subplot(222, projection='3d')
    ax3 = fig.add_subplot(223, projection='3d')
    ax4 = fig.add_subplot(224, projection='3d')
    
    # Hybrid1 (Dataset img + Robot state) - Left arm
    for i in range(NUM_SAMPLES):
        left_traj, _ = extract_trajectory(actions_hybrid1[i], obs_hybrid_dataset_img_robot_state, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax1.plot(left_traj[:, 0], left_traj[:, 1], left_traj[:, 2], 
                 'g-o' if i == 0 else 'g-', linewidth=lw, markersize=2, alpha=alpha)
    ax1.set_title('Dataset Img + Robot State - Left', fontsize=11, fontweight='bold', color='green')
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    
    # Hybrid1 - Right arm
    for i in range(NUM_SAMPLES):
        _, right_traj = extract_trajectory(actions_hybrid1[i], obs_hybrid_dataset_img_robot_state, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax2.plot(right_traj[:, 0], right_traj[:, 1], right_traj[:, 2], 
                 'g-o' if i == 0 else 'g-', linewidth=lw, markersize=2, alpha=alpha)
    ax2.set_title('Dataset Img + Robot State - Right', fontsize=11, fontweight='bold', color='green')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    
    # Hybrid2 (Robot img + Dataset state) - Left arm
    for i in range(NUM_SAMPLES):
        left_traj, _ = extract_trajectory(actions_hybrid2[i], obs_hybrid_robot_img_dataset_state, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax3.plot(left_traj[:, 0], left_traj[:, 1], left_traj[:, 2], 
                 'm-o' if i == 0 else 'm-', linewidth=lw, markersize=2, alpha=alpha)
    ax3.set_title('Robot Img + Dataset State - Left', fontsize=11, fontweight='bold', color='purple')
    ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')
    
    # Hybrid2 - Right arm
    for i in range(NUM_SAMPLES):
        _, right_traj = extract_trajectory(actions_hybrid2[i], obs_hybrid_robot_img_dataset_state, action_mode)
        lw = 3 if i == 0 else 1
        alpha = 1.0 if i == 0 else 0.4
        ax4.plot(right_traj[:, 0], right_traj[:, 1], right_traj[:, 2], 
                 'm-o' if i == 0 else 'm-', linewidth=lw, markersize=2, alpha=alpha)
    ax4.set_title('Robot Img + Dataset State - Right', fontsize=11, fontweight='bold', color='purple')
    ax4.set_xlabel('X'); ax4.set_ylabel('Y'); ax4.set_zlabel('Z')
    
    plt.suptitle(f'HYBRID Action Chunks @ Frame {current_frame}\nGreen=DatasetImg+RobotState, Purple=RobotImg+DatasetState', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 3: All 4 sources overlay comparison
    # ============================================================
    fig = plt.figure(figsize=(14, 6))
    
    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')
    
    # Extract trajectories for Sample 0 from each source
    left_d, right_d = extract_trajectory(actions_dataset[0], obs_dataset, action_mode)
    left_r, right_r = extract_trajectory(actions_robot[0], obs_robot, action_mode)
    left_h1, right_h1 = extract_trajectory(actions_hybrid1[0], obs_hybrid_dataset_img_robot_state, action_mode)
    left_h2, right_h2 = extract_trajectory(actions_hybrid2[0], obs_hybrid_robot_img_dataset_state, action_mode)
    
    # Left arm overlay
    ax1.plot(left_d[:, 0], left_d[:, 1], left_d[:, 2], 'b-o', linewidth=2, markersize=4, label='Dataset', alpha=0.8)
    ax1.plot(left_r[:, 0], left_r[:, 1], left_r[:, 2], 'r-s', linewidth=2, markersize=4, label='Robot', alpha=0.8)
    ax1.plot(left_h1[:, 0], left_h1[:, 1], left_h1[:, 2], 'g-^', linewidth=2, markersize=4, label='DatImg+RobState', alpha=0.8)
    ax1.plot(left_h2[:, 0], left_h2[:, 1], left_h2[:, 2], 'm-d', linewidth=2, markersize=4, label='RobImg+DatState', alpha=0.8)
    ax1.set_title('Left Arm - All Sources')
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    ax1.legend(fontsize=8)
    
    # Right arm overlay
    ax2.plot(right_d[:, 0], right_d[:, 1], right_d[:, 2], 'b-o', linewidth=2, markersize=4, label='Dataset', alpha=0.8)
    ax2.plot(right_r[:, 0], right_r[:, 1], right_r[:, 2], 'r-s', linewidth=2, markersize=4, label='Robot', alpha=0.8)
    ax2.plot(right_h1[:, 0], right_h1[:, 1], right_h1[:, 2], 'g-^', linewidth=2, markersize=4, label='DatImg+RobState', alpha=0.8)
    ax2.plot(right_h2[:, 0], right_h2[:, 1], right_h2[:, 2], 'm-d', linewidth=2, markersize=4, label='RobImg+DatState', alpha=0.8)
    ax2.set_title('Right Arm - All Sources')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    ax2.legend(fontsize=8)
    
    plt.suptitle(f'All 4 Sources Overlay (Sample 0) @ Frame {current_frame}')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    
    # ============================================================
    # Page 4: Action values as text (all 4 sources) - NO EMOJIS for PDF
    # ============================================================
    action_d = actions_dataset[0].detach().cpu().numpy()
    action_r = actions_robot[0].detach().cpu().numpy()
    action_h1 = actions_hybrid1[0].detach().cpu().numpy()
    action_h2 = actions_hybrid2[0].detach().cpu().numpy()
    
    fig = plt.figure(figsize=(14, 16))
    ax = fig.add_subplot(111)
    ax.axis('off')
    
    # Build text content - NO EMOJIS (use plain text for PDF compatibility)
    lines = []
    lines.append(f"ACTION VALUES (Sample 0) @ Frame {current_frame}")
    lines.append("=" * 90)
    lines.append(f"Shape: {action_d.shape} (n_action_steps, action_dim)")
    lines.append(f"Action mode: {action_mode}")
    lines.append("")
    
    lines.append("-" * 90)
    lines.append("[BLUE] DATASET (full):")
    lines.append("-" * 90)
    for t in range(min(action_d.shape[0], 8)):
        lines.append(f"  Step {t:2d}: {np.array2string(action_d[t], precision=4, suppress_small=True, max_line_width=200)}")
    if action_d.shape[0] > 8:
        lines.append(f"  ... ({action_d.shape[0] - 8} more steps)")
    
    lines.append("")
    lines.append("-" * 90)
    lines.append("[RED] ROBOT (full):")
    lines.append("-" * 90)
    for t in range(min(action_r.shape[0], 8)):
        lines.append(f"  Step {t:2d}: {np.array2string(action_r[t], precision=4, suppress_small=True, max_line_width=200)}")
    if action_r.shape[0] > 8:
        lines.append(f"  ... ({action_r.shape[0] - 8} more steps)")
    
    lines.append("")
    lines.append("-" * 90)
    lines.append("[GREEN] HYBRID1 (Dataset Img + Robot State):")
    lines.append("-" * 90)
    for t in range(min(action_h1.shape[0], 8)):
        lines.append(f"  Step {t:2d}: {np.array2string(action_h1[t], precision=4, suppress_small=True, max_line_width=200)}")
    if action_h1.shape[0] > 8:
        lines.append(f"  ... ({action_h1.shape[0] - 8} more steps)")
    
    lines.append("")
    lines.append("-" * 90)
    lines.append("[PURPLE] HYBRID2 (Robot Img + Dataset State):")
    lines.append("-" * 90)
    for t in range(min(action_h2.shape[0], 8)):
        lines.append(f"  Step {t:2d}: {np.array2string(action_h2[t], precision=4, suppress_small=True, max_line_width=200)}")
    if action_h2.shape[0] > 8:
        lines.append(f"  ... ({action_h2.shape[0] - 8} more steps)")
    
    text = "\n".join(lines)
    ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=7, fontfamily='monospace',
            verticalalignment='top', horizontalalignment='left')
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

print(f"\n💾 Saved complete report: {pdf_path}")

# ============================================================
# Print summary to console (emojis OK here - not saved to PDF)
# ============================================================
print("\n" + "=" * 80)
print("📊 SUMMARY: Mean action differences (Sample 0)")
print("=" * 80)

diff_dataset_robot = np.abs(action_d - action_r).mean()
diff_dataset_hybrid1 = np.abs(action_d - action_h1).mean()
diff_dataset_hybrid2 = np.abs(action_d - action_h2).mean()
diff_robot_hybrid1 = np.abs(action_r - action_h1).mean()
diff_robot_hybrid2 = np.abs(action_r - action_h2).mean()
diff_hybrid1_hybrid2 = np.abs(action_h1 - action_h2).mean()

print(f"\n   Dataset vs Robot:                    {diff_dataset_robot:.6f}")
print(f"   Dataset vs Hybrid1 (DatImg+RobState): {diff_dataset_hybrid1:.6f}")
print(f"   Dataset vs Hybrid2 (RobImg+DatState): {diff_dataset_hybrid2:.6f}")
print(f"   Robot vs Hybrid1 (DatImg+RobState):   {diff_robot_hybrid1:.6f}")
print(f"   Robot vs Hybrid2 (RobImg+DatState):   {diff_robot_hybrid2:.6f}")
print(f"   Hybrid1 vs Hybrid2:                   {diff_hybrid1_hybrid2:.6f}")

print("\n" + "─" * 80)
print("🔍 INTERPRETATION:")
print("─" * 80)
if diff_dataset_hybrid2 < diff_dataset_robot * 0.5:
    print("   → State drives most of the difference (Hybrid2 ≈ Dataset)")
elif diff_dataset_hybrid1 < diff_dataset_robot * 0.5:
    print("   → Images drive most of the difference (Hybrid1 ≈ Dataset)")
else:
    print("   → Both images and state contribute to the difference")

print("\n" + "=" * 80)

## 9. Quantitative Comparison

In [ ]:
# Compare action chunks numerically
actions_d_np = actions_dataset.detach().cpu().numpy()
actions_r_np = actions_robot.detach().cpu().numpy()

print("=" * 60)
print("📈 QUANTITATIVE COMPARISON")
print("=" * 60)

# Within-source variance (how diverse are samples from same source)
dataset_var = actions_d_np.std(axis=0).mean()
robot_var = actions_r_np.std(axis=0).mean()

print(f"\n🎯 Within-Source Variance (sample diversity):")
print(f"   Dataset samples std: {dataset_var:.6f}")
print(f"   Robot samples std:   {robot_var:.6f}")

# Cross-source difference (how different are dataset vs robot predictions)
mean_dataset = actions_d_np.mean(axis=0)  # (n_action_steps, action_dim)
mean_robot = actions_r_np.mean(axis=0)

cross_diff = np.abs(mean_dataset - mean_robot)

print(f"\n🔄 Cross-Source Difference (dataset mean vs robot mean):")
print(f"   Mean abs diff: {cross_diff.mean():.6f}")
print(f"   Max abs diff:  {cross_diff.max():.6f}")
print(f"   Frobenius norm: {np.linalg.norm(mean_dataset - mean_robot):.6f}")

# Position trajectory differences
left_d_mean, right_d_mean = extract_trajectory(torch.tensor(mean_dataset), obs_dataset, action_mode)
left_r_mean, right_r_mean = extract_trajectory(torch.tensor(mean_robot), obs_robot, action_mode)

left_pos_diff = np.linalg.norm(left_d_mean - left_r_mean, axis=1).mean()
right_pos_diff = np.linalg.norm(right_d_mean - right_r_mean, axis=1).mean()

print(f"\n📍 Position Trajectory Difference (mean trajectories):")
print(f"   Left arm mean position diff:  {left_pos_diff:.6f} m")
print(f"   Right arm mean position diff: {right_pos_diff:.6f} m")

# Endpoint difference
left_endpoint_diff = np.linalg.norm(left_d_mean[-1] - left_r_mean[-1])
right_endpoint_diff = np.linalg.norm(right_d_mean[-1] - right_r_mean[-1])

print(f"\n🎯 Endpoint Difference:")
print(f"   Left arm endpoint diff:  {left_endpoint_diff:.6f} m")
print(f"   Right arm endpoint diff: {right_endpoint_diff:.6f} m")

print("\n" + "=" * 60)

## 10. Iterate Through Episode

Step through the episode and compare at each frame.

In [ ]:
# Advance to next frame and compare
current_frame += 1

if current_frame >= episode_length:
    print(f"⚠️ Reached end of episode! Resetting to frame 0.")
    current_frame = 0

print(f"📍 Frame {current_frame}/{episode_length-1}")
print("=" * 60)

# Get new observations
obs_dataset = get_observation_from_dataset(current_frame)
obs_robot = get_observation_from_robot()

# Quick state comparison
state_d, state_r, state_diff = compare_states(obs_dataset, obs_robot)

# Generate and compare actions
policy.reset()
actions_dataset = generate_action_samples(obs_dataset, NUM_SAMPLES)
actions_robot = generate_action_samples(obs_robot, NUM_SAMPLES)

# Quick overlay plot
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

left_d, right_d = extract_trajectory(actions_dataset[0], obs_dataset, action_mode)
left_r, right_r = extract_trajectory(actions_robot[0], obs_robot, action_mode)

ax1.plot(left_d[:, 0], left_d[:, 1], left_d[:, 2], 'b-o', linewidth=2, markersize=3, label='Dataset')
ax1.plot(left_r[:, 0], left_r[:, 1], left_r[:, 2], 'r-s', linewidth=2, markersize=3, label='Robot')
ax1.set_title('Left Arm'); ax1.legend()

ax2.plot(right_d[:, 0], right_d[:, 1], right_d[:, 2], 'b-o', linewidth=2, markersize=3, label='Dataset')
ax2.plot(right_r[:, 0], right_r[:, 1], right_r[:, 2], 'r-s', linewidth=2, markersize=3, label='Robot')
ax2.set_title('Right Arm'); ax2.legend()

plt.suptitle(f'Frame {current_frame}: Dataset (blue) vs Robot (red)')
plt.tight_layout()
plt.show()

# Quick stats
cross_diff = np.abs(actions_dataset[0].cpu().numpy() - actions_robot[0].cpu().numpy())
print(f"\n📊 Action diff (sample 0): mean={cross_diff.mean():.6f}, max={cross_diff.max():.6f}")

## 11. Jump to Specific Frame

In [ ]:
# Set specific frame to analyze
current_frame = 50  # <-- Change this value

if current_frame >= episode_length:
    current_frame = episode_length - 1
    print(f"⚠️ Clamped to max frame {current_frame}")

print(f"📍 Jumping to frame {current_frame}/{episode_length-1}")

# Now run cell 10 or 8 to compare at this frame

## 12. Cleanup

In [ ]:
# Close robot connection
try:
    robot_connection.__exit__(None, None, None)
    print("✅ Robot connection closed.")
except:
    pass